# 03 — Modelado base y decisiones sensibles al costo

Este notebook implementa el primer pipeline temporal del proyecto: preprocesamiento ajustado en train, un predictor probabilístico base, calibración en una ventana temprana de validation y evaluación predictiva en una ventana posterior.

**Estado del protocolo económico:** la matriz y las fórmulas están propuestas en `docs/03_protocolo_costos_y_evaluacion.md`, pero el equipo todavía debe aprobar los parámetros de costo. Por eso el notebook permite obtener métricas predictivas de validation, pero mantiene bloqueados los costos/ahorros y toda predicción/evaluación de test mientras falten aprobaciones. BMR binario queda disponible cuando se aprueban solo los costos base; revisión requiere `REVIEW_ENABLED=True` y sus propios parámetros. El árbol ponderado se omite si el costo residual de fraude bloqueado no es cero. No se usan valores económicos ficticios.

El predictor XGBoost de este notebook es un baseline técnico predeclarado, no una conclusión de que sea el mejor modelo. Los hiperparámetros permanecen fijos; no se ajustan mirando validation.

## 1. Configuración y prerrequisitos

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from fraud_cost.data import load_ieee_cis
from fraud_cost.split import validate_temporal_split


def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Abra Jupyter desde el repositorio o una de sus subcarpetas")


PROJECT_ROOT = find_project_root()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
RESULTS_TABLES = PROJECT_ROOT / "results" / "tables"
RESULTS_FIGURES = PROJECT_ROOT / "results" / "figures"
RESULTS_MODELS = PROJECT_ROOT / "results" / "models"
for directory in (DATA_INTERIM, RESULTS_TABLES, RESULTS_FIGURES, RESULTS_MODELS):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
# Propuesta inicial: mitad temprana de validation para calibrar, mitad posterior
# para medir/seleccionar políticas. Registrar cualquier cambio antes de ejecutar.
VALIDATION_CALIBRATION_FRACTION = 0.50
MIN_CATEGORY_FREQUENCY = 50

# Baseline fijo y predeclarado; estos parámetros no se optimizan con validation.
BASE_MODEL_PARAMS = {
    "n_estimators": 250,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 10,
    "tree_method": "hist",
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "n_jobs": -1,
    "random_state": RANDOM_STATE,
    "verbosity": 0,
}

# Completar solo con parámetros económicos acordados y documentados.
REVIEW_ENABLED = False  # Habilitar solo cuando el equipo apruebe todos los parámetros de revisión.

COST_CONFIG = {
    "approved": False,
    "approval_reference": None,
    "amount_unit": None,
    "loss_fraction": None,                 # lambda
    "legitimate_block_cost": None,          # B (constante por fila)
    "fraud_block_residual_cost": None,      # C_block_fraud
    "review_cost_grid": None,               # valores R predeclarados
    "review_sensitivity": None,             # s_R
    "review_specificity": None,             # q_R
    "review_capacity_per_day": None,        # entero o "unlimited", aprobado explícitamente
    "review_day_origin_transactiondt": None, # origen fijo aprobado si existe capacidad diaria
}

# Solo activar después de aprobar costos y congelar el protocolo/modelos.
RUN_FINAL_TEST = False
FINAL_TEST_PROTOCOL_FROZEN = False

if not 0 < VALIDATION_CALIBRATION_FRACTION < 1:
    raise ValueError("VALIDATION_CALIBRATION_FRACTION debe estar entre 0 y 1")
print(f"Proyecto: {PROJECT_ROOT}")
print(f"Python baseline: XGBoost; parámetros fijos: {BASE_MODEL_PARAMS}")
print(f"Validation: {VALIDATION_CALIBRATION_FRACTION:.0%} temprano para calibración")

## 2. Cargar datos y manifiesto temporal

In [ ]:
transactions, identity, merged = load_ieee_cis(DATA_RAW)
manifest_path = DATA_INTERIM / "split_manifest.parquet"
if not manifest_path.exists():
    raise FileNotFoundError(
        f"No existe {manifest_path}; ejecute primero notebooks/02_particion_temporal.ipynb"
    )
manifest = pd.read_parquet(manifest_path, columns=["TransactionID", "split"])
if manifest["TransactionID"].duplicated().any():
    raise ValueError("El manifiesto contiene TransactionID duplicados")

identity_ids = set(identity["TransactionID"])
merged["has_identity"] = merged["TransactionID"].isin(identity_ids).astype("int8")
frame = merged.merge(
    manifest,
    on="TransactionID",
    how="left",
    validate="one_to_one",
)
if len(frame) != len(transactions) or frame["split"].isna().any():
    raise AssertionError("El manifiesto no cubre exactamente las transacciones")
frame = frame.sort_values(["TransactionDT", "TransactionID"]).reset_index(drop=True)
validate_temporal_split(frame, frame["split"])

split_summary = frame.groupby("split", observed=True).agg(
    rows=("TransactionID", "size"),
    frauds=("isFraud", "sum"),
    fraud_rate=("isFraud", "mean"),
)
print(split_summary.reindex(["train", "gap_1", "validation", "gap_2", "test"]))
assert set(frame["split"].unique()) == {"train", "gap_1", "validation", "gap_2", "test"}

## 3. Ventanas de train y validation; preparación sin fuga

In [ ]:
# Los gaps se excluyen. Test no se extrae ni se puntúa en estas celdas.
train = frame.loc[frame["split"].eq("train")].copy()
validation = frame.loc[frame["split"].eq("validation")].copy()
validation = validation.sort_values(["TransactionDT", "TransactionID"]).reset_index(drop=True)
calibration_end = int(round(len(validation) * VALIDATION_CALIBRATION_FRACTION))
validation_cal = validation.iloc[:calibration_end].copy()
validation_policy = validation.iloc[calibration_end:].copy()

for name, block in (("train", train), ("validation_cal", validation_cal), ("validation_policy", validation_policy)):
    if block.empty or block["isFraud"].nunique() < 2:
        raise ValueError(f"{name} debe contener ambas clases para este protocolo")
    print(name, "rows=", len(block), "frauds=", int(block["isFraud"].sum()),
          "time_start=", int(block["TransactionDT"].min()),
          "time_end=", int(block["TransactionDT"].max()))

EXCLUDED_FEATURES = {"TransactionID", "TransactionDT", "isFraud", "split"}
CATEGORICAL_HINTS = {
    "ProductCD", "card1", "card2", "card3", "card4", "card5", "card6",
    "addr1", "addr2", "P_emaildomain", "R_emaildomain", "DeviceType", "DeviceInfo",
    *(f"M{i}" for i in range(1, 10)),
}


def make_features(data):
    X = data.drop(columns=[c for c in EXCLUDED_FEATURES if c in data.columns]).copy()
    for column in CATEGORICAL_HINTS.intersection(X.columns):
        X[column] = X[column].astype("string").fillna("__MISSING__")
    X = X.replace([np.inf, -np.inf], np.nan)
    return X


X_train = make_features(train)
X_cal = make_features(validation_cal)
X_policy = make_features(validation_policy)
y_train = train["isFraud"].astype("int8")
y_cal = validation_cal["isFraud"].astype("int8")
y_policy = validation_policy["isFraud"].astype("int8")

categorical_columns = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
numeric_columns = [column for column in X_train.columns if column not in categorical_columns]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=MIN_CATEGORY_FREQUENCY,
        sparse_output=True,
        dtype=np.float32,
    )),
])
preprocessor = ColumnTransformer(
    [("numeric", numeric_pipeline, numeric_columns),
     ("categorical", categorical_pipeline, categorical_columns)],
    remainder="drop",
    sparse_threshold=0.3,
)

# Vocabulario, imputación y cualquier estadística se aprenden solo con train.
X_train_t = preprocessor.fit_transform(X_train)
X_cal_t = preprocessor.transform(X_cal)
X_policy_t = preprocessor.transform(X_policy)
print("Matriz train:", X_train_t.shape, "; variables numéricas:", len(numeric_columns),
      "; categóricas:", len(categorical_columns))

## 4. Predictor base y calibración temporal

In [ ]:
base_model = XGBClassifier(**BASE_MODEL_PARAMS)
base_model.fit(X_train_t, y_train, verbose=False)


def clipped_logit(probabilities):
    p = np.clip(np.asarray(probabilities, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p)).reshape(-1, 1)


# Calibración Platt: el predictor no vio validation_cal ni validation_policy.
p_cal_raw = base_model.predict_proba(X_cal_t)[:, 1]
calibrator = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1000)
calibrator.fit(clipped_logit(p_cal_raw), y_cal)
p_policy_raw = base_model.predict_proba(X_policy_t)[:, 1]
p_policy = calibrator.predict_proba(clipped_logit(p_policy_raw))[:, 1]

# Baseline fijo documentado; el umbral no se optimiza con validation.
action_fixed_policy = np.where(p_policy >= 0.5, "block", "approve")
print("Predictor entrenado con train; calibrador ajustado en validation temprana.")
print("No se han calculado acciones ni métricas de test.")

## 5. Métricas predictivas de validation posterior

In [ ]:
validation_metrics = pd.DataFrame([{
    "block": "validation_policy",
    "rows": len(y_policy),
    "frauds": int(y_policy.sum()),
    "fraud_rate": float(y_policy.mean()),
    "average_precision": float(average_precision_score(y_policy, p_policy)),
    "recall_at_fixed_0_5": float(recall_score(y_policy, action_fixed_policy == "block")),
    "precision_at_fixed_0_5": float(precision_score(y_policy, action_fixed_policy == "block", zero_division=0)),
    "brier_score": float(brier_score_loss(y_policy, p_policy)),
    "log_loss": float(log_loss(y_policy, p_policy, labels=[0, 1])),
    "legitimate_block_rate_at_fixed_0_5": float(
        np.mean(action_fixed_policy[y_policy.to_numpy() == 0] == "block")
    ),
}])
RESULTS_TABLES.joinpath("modelo_metricas_validation.csv").parent.mkdir(parents=True, exist_ok=True)
validation_metrics.to_csv(RESULTS_TABLES / "modelo_metricas_validation.csv", index=False)

fraction_positive, mean_predicted = calibration_curve(y_policy, p_policy, n_bins=10, strategy="quantile")
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(mean_predicted, fraction_positive, marker="o", label="XGBoost + Platt")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Calibración ideal")
ax.set(xlabel="Probabilidad promedio", ylabel="Fracción observada de fraude",
       title="Calibración en validation posterior", xlim=(0, 1), ylim=(0, 1))
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_FIGURES / "modelo_calibracion_validation.png", dpi=160, bbox_inches="tight")
plt.show()

validation_scores = validation_policy[["TransactionID", "TransactionDT", "TransactionAmt", "isFraud"]].copy()
validation_scores["probability_raw"] = p_policy_raw
validation_scores["probability_calibrated"] = p_policy
validation_scores["action_fixed_0_5"] = action_fixed_policy
validation_scores.to_parquet(DATA_INTERIM / "validation_scores.parquet", index=False)
joblib.dump(
    {"preprocessor": preprocessor, "model": base_model, "calibrator": calibrator,
     "base_model_params": BASE_MODEL_PARAMS,
     "validation_calibration_fraction": VALIDATION_CALIBRATION_FRACTION},
    RESULTS_MODELS / "baseline_xgboost_platt.joblib",
)
display(validation_metrics)

## 6. Funciones de costo y gate de aprobación

In [ ]:
REQUIRED_COST_FIELDS = (
    "approval_reference", "amount_unit", "loss_fraction",
    "legitimate_block_cost", "fraud_block_residual_cost",

)


def missing_cost_fields(config):
    missing = [key for key in REQUIRED_COST_FIELDS if config.get(key) is None]
    if REVIEW_ENABLED:
        review_fields = ("review_cost_grid", "review_sensitivity", "review_specificity", "review_capacity_per_day")
        missing.extend(key for key in review_fields if config.get(key) is None)
        if config.get("review_cost_grid") is not None and len(config["review_cost_grid"]) == 0:
            missing.append("review_cost_grid (al menos un escenario predeclarado)")
        capacity = config.get("review_capacity_per_day")
        if capacity not in (None, "unlimited") and config.get("review_day_origin_transactiondt") is None:
            missing.append("review_day_origin_transactiondt (origen fijo para cupos diarios)")
    return missing


def require_approved_costs(config):
    missing = missing_cost_fields(config)
    if not config.get("approved", False):
        missing.append("approved=True (aprobación del equipo)")
    if missing:
        raise RuntimeError("Evaluación económica bloqueada. Complete/apruebe: " + ", ".join(missing))
    if not isinstance(config["approval_reference"], str) or not config["approval_reference"].strip():
        raise ValueError("Registre quién/cuándo aprobó los supuestos")
    core_values = [config["loss_fraction"], config["legitimate_block_cost"], config["fraud_block_residual_cost"]]
    if any(not np.isfinite(float(value)) or float(value) < 0 for value in core_values):
        raise ValueError("Los costos base deben ser finitos y no negativos")
    if not isinstance(config["amount_unit"], str) or not config["amount_unit"].strip():
        raise ValueError("amount_unit debe identificar la unidad aprobada")
    if float(config["fraud_block_residual_cost"]) < 0:
        raise ValueError("fraud_block_residual_cost no puede ser negativo")
    if REVIEW_ENABLED:
        if not np.isfinite(float(config["review_sensitivity"])) or not 0 <= float(config["review_sensitivity"]) <= 1:
            raise ValueError("review_sensitivity debe ser finita y estar entre 0 y 1")
        if not np.isfinite(float(config["review_specificity"])) or not 0 <= float(config["review_specificity"]) <= 1:
            raise ValueError("review_specificity debe ser finita y estar entre 0 y 1")
        capacity = config["review_capacity_per_day"]
        if capacity != "unlimited" and (not isinstance(capacity, (int, np.integer)) or capacity < 0):
            raise ValueError("review_capacity_per_day debe ser entero >=0 o 'unlimited'")
        if any(not np.isfinite(float(value)) or float(value) < 0 for value in config["review_cost_grid"]):
            raise ValueError("Los costos de revisión deben ser finitos y no negativos")
        if config["amount_unit"].strip() == "":
            raise ValueError("amount_unit debe estar definido")


def bmr_actions(probability, amount, transaction_time, config, review_cost):
    p = np.clip(np.asarray(probability, dtype=float), 0, 1)
    a = np.asarray(amount, dtype=float)
    risk_approve = p * float(config["loss_fraction"]) * a
    risk_block = ((1 - p) * float(config["legitimate_block_cost"])
                  + p * float(config["fraud_block_residual_cost"]))
    if not REVIEW_ENABLED:
        return np.where(risk_approve <= risk_block, "approve", "block")
    s = float(config["review_sensitivity"])
    q = float(config["review_specificity"])
    risk_review = (float(review_cost)
                   + p * ((1 - s) * float(config["loss_fraction"]) * a
                          + s * float(config["fraud_block_residual_cost"]))
                   + (1 - p) * (1 - q) * float(config["legitimate_block_cost"]))
    # Empates: no revisar; entre aprobar y bloquear, aprobar.
    actions = np.where(risk_approve <= risk_block, "approve", "block")
    capacity = config["review_capacity_per_day"]
    if capacity == "unlimited":
        risk_stack = np.vstack([risk_approve, risk_block, risk_review])
        actions = np.array(["approve", "block", "review"])[np.argmin(risk_stack, axis=0)]
    else:
        benefit = np.minimum(risk_approve, risk_block) - risk_review
        origin = int(config["review_day_origin_transactiondt"])
        days = ((np.asarray(transaction_time) - origin) // 86400).astype(int)
        for day in np.unique(days):
            ids = np.flatnonzero((days == day) & (benefit > 0))
            chosen = ids[np.argsort(-benefit[ids], kind="stable")[:int(capacity)]]
            actions[chosen] = "review"
    return actions


def realized_expected_cost(y_true, amount, actions, config, review_cost):
    y = np.asarray(y_true, dtype=int)
    a = np.asarray(amount, dtype=float)
    actions = np.asarray(actions)
    cost = np.zeros(len(y), dtype=float)
    fraud = y == 1
    approve = actions == "approve"
    block = actions == "block"
    review = actions == "review"
    cost[fraud & approve] = float(config["loss_fraction"]) * a[fraud & approve]
    cost[fraud & block] = float(config["fraud_block_residual_cost"])
    cost[(~fraud) & block] = float(config["legitimate_block_cost"])
    if review.any():
        if not REVIEW_ENABLED or review_cost is None:
            raise ValueError("Hay acciones de revisión sin parámetros aprobados")
        cost[review] = float(review_cost)
    if review.any():
        cost[review & fraud] += (
        (1 - float(config["review_sensitivity"]))
        * float(config["loss_fraction"]) * a[review & fraud]
        + float(config["review_sensitivity"])
        * float(config["fraud_block_residual_cost"])
    )
    cost[review & (~fraud)] += (
        (1 - float(config["review_specificity"]))
        * float(config["legitimate_block_cost"])
    )
    return cost

## 7. Evaluación económica en validation (solo tras aprobación)

In [ ]:
if not COST_CONFIG.get("approved", False):
    print("BLOQUEADO: no hay costos aprobados. Se conservaron únicamente métricas predictivas de validation.")
else:
    require_approved_costs(COST_CONFIG)
    # La ponderación binaria documentada es válida como candidato con costo residual de fraude bloqueado = 0.
    action_tree = None
    cost_tree = None
    if float(COST_CONFIG["fraud_block_residual_cost"]) == 0:
        train_amount = train["TransactionAmt"].to_numpy(dtype=float)
        sample_weight = np.where(
            y_train.to_numpy() == 1,
            float(COST_CONFIG["loss_fraction"]) * train_amount,
            float(COST_CONFIG["legitimate_block_cost"]),
        )
        if not np.isfinite(sample_weight).all() or sample_weight.sum() <= 0:
            raise ValueError("sample_weight inválido; revise costos y montos")
        sample_weight = sample_weight / sample_weight.mean()
        cost_tree = DecisionTreeClassifier(
            max_depth=12, min_samples_leaf=250, random_state=RANDOM_STATE
        )
        cost_tree.fit(X_train_t, y_train, sample_weight=sample_weight)
        action_tree = np.where(cost_tree.predict(X_policy_t) == 1, "block", "approve")
        joblib.dump(cost_tree, RESULTS_MODELS / "cost_sensitive_tree.joblib")
    else:
        print("Árbol ponderado omitido: esta aproximación requiere fraud_block_residual_cost=0; fixed y BMR se mantienen disponibles.")

    cost_rows = []
    amount_policy = validation_policy["TransactionAmt"].to_numpy(dtype=float)
    time_policy = validation_policy["TransactionDT"].to_numpy(dtype=np.int64)
    review_costs = COST_CONFIG["review_cost_grid"] if REVIEW_ENABLED else [None]
    for review_cost in review_costs:
        action_bmr = bmr_actions(p_policy, amount_policy, time_policy, COST_CONFIG, review_cost)
        strategies = (("fixed_0_5", action_fixed_policy), ("bmr", action_bmr))
        if action_tree is not None:
            strategies = strategies + (("cost_sensitive_tree", action_tree),)
        for strategy, actions in strategies:
            row_cost = realized_expected_cost(
                y_policy, amount_policy, actions, COST_CONFIG, review_cost
            )
            fraud_mask = y_policy.to_numpy() == 1
            legit_mask = ~fraud_mask
            reviewed_fraud = fraud_mask & (actions == "review")
            blocked_fraud = fraud_mask & (actions == "block")
            fraud_amount = amount_policy[fraud_mask].sum()
            expected_detected_amount = (
                amount_policy[blocked_fraud].sum()
                + (float(COST_CONFIG["review_sensitivity"]) if REVIEW_ENABLED else 0.0) * amount_policy[reviewed_fraud].sum()
            )
            fixed_cost_proxy = float(realized_expected_cost(
                y_policy, amount_policy, action_fixed_policy, COST_CONFIG, review_cost
            ).sum())
            strategy_cost_proxy = float(row_cost.sum())
            cost_rows.append({
                "validation_block": "validation_policy",
                "strategy": strategy,
                "review_cost": review_cost,
                "cost_total_proxy": strategy_cost_proxy,
                "cost_per_transaction_proxy": float(row_cost.mean()),
                "fixed_baseline_cost_proxy": fixed_cost_proxy,
                "savings_vs_fixed_proxy": fixed_cost_proxy - strategy_cost_proxy,
                "savings_fraction_vs_fixed_proxy": ((fixed_cost_proxy - strategy_cost_proxy) / fixed_cost_proxy if fixed_cost_proxy > 0 else np.nan),
                "blocked_legitimate_count": int(np.sum(legit_mask & (actions == "block"))),
                "blocked_legitimate_rate": float(np.mean(actions[legit_mask] == "block")),
                "review_rate": float(np.mean(actions == "review")),
                "frauds_blocked": int(blocked_fraud.sum()),
                "frauds_reviewed": int(reviewed_fraud.sum()),
                "expected_fraud_recall": float((blocked_fraud.sum() + (float(COST_CONFIG["review_sensitivity"]) if REVIEW_ENABLED else 0.0) * reviewed_fraud.sum()) / fraud_mask.sum()),
                "expected_amount_weighted_recall": float(expected_detected_amount / fraud_amount),
            })
    validation_costs = pd.DataFrame(cost_rows)
    validation_costs.to_csv(RESULTS_TABLES / "costos_validation_sensibilidad.csv", index=False)
    display(validation_costs)

    print("Estos son resultados de selección/diagnóstico en validation, no evaluación final.")

## 8. Test final: gate explícito

In [ ]:
if not RUN_FINAL_TEST:
    print("Test no puntuado. Mantenga RUN_FINAL_TEST=False hasta aprobar costos y congelar el protocolo.")
else:
    require_approved_costs(COST_CONFIG)
    if not FINAL_TEST_PROTOCOL_FROZEN:
        raise RuntimeError("Congele y registre modelos, calibrador, costos, escenarios y reglas antes de consultar test")

    # Esta es la primera celda que extrae, transforma y puntúa filas de test.
    test = frame.loc[frame["split"].eq("test")].copy()
    X_test = make_features(test)
    y_test = test["isFraud"].astype("int8")
    X_test_t = preprocessor.transform(X_test)
    p_test_raw = base_model.predict_proba(X_test_t)[:, 1]
    p_test = calibrator.predict_proba(clipped_logit(p_test_raw))[:, 1]
    action_fixed_test = np.where(p_test >= 0.5, "block", "approve")
    action_tree_test = (np.where(cost_tree.predict(X_test_t) == 1, "block", "approve")
                        if cost_tree is not None else None)

    test_cost_rows = []
    amount_test = test["TransactionAmt"].to_numpy(dtype=float)
    time_test = test["TransactionDT"].to_numpy(dtype=np.int64)
    review_costs = COST_CONFIG["review_cost_grid"] if REVIEW_ENABLED else [None]
    for review_cost in review_costs:
        action_bmr_test = bmr_actions(p_test, amount_test, time_test, COST_CONFIG, review_cost)
        strategies = (("fixed_0_5", action_fixed_test), ("bmr", action_bmr_test))
        if action_tree_test is not None:
            strategies = strategies + (("cost_sensitive_tree", action_tree_test),)
        for strategy, actions in strategies:
            row_cost = realized_expected_cost(y_test, amount_test, actions, COST_CONFIG, review_cost)
            fixed_cost_proxy = float(realized_expected_cost(
                y_test, amount_test, action_fixed_test, COST_CONFIG, review_cost
            ).sum())
            strategy_cost_proxy = float(row_cost.sum())
            fraud_mask_test = y_test.to_numpy() == 1
            legit_mask_test = ~fraud_mask_test
            blocked_fraud_test = fraud_mask_test & (actions == "block")
            reviewed_fraud_test = fraud_mask_test & (actions == "review")
            fraud_amount_test = amount_test[fraud_mask_test].sum()
            expected_detected_amount_test = (amount_test[blocked_fraud_test].sum() + (float(COST_CONFIG["review_sensitivity"]) if REVIEW_ENABLED else 0.0) * amount_test[reviewed_fraud_test].sum())
            test_cost_rows.append({
                "strategy": strategy,
                "review_cost": review_cost,
                "cost_total_proxy": strategy_cost_proxy,
                "cost_per_transaction_proxy": float(row_cost.mean()),
                "fixed_baseline_cost_proxy": fixed_cost_proxy,
                "savings_vs_fixed_proxy": fixed_cost_proxy - strategy_cost_proxy,
                "savings_fraction_vs_fixed_proxy": ((fixed_cost_proxy - strategy_cost_proxy) / fixed_cost_proxy if fixed_cost_proxy > 0 else np.nan),
                "base_model_average_precision_common": float(average_precision_score(y_test, p_test)),
                "base_model_recall_fixed_0_5_common": float(recall_score(y_test, action_fixed_test == "block")),
                "blocked_legitimate_count": int(np.sum(legit_mask_test & (actions == "block"))),
                "blocked_legitimate_rate": float(np.mean(actions[legit_mask_test] == "block")),
                "frauds_blocked": int(blocked_fraud_test.sum()),
                "frauds_reviewed": int(reviewed_fraud_test.sum()),
                "expected_fraud_recall": float((blocked_fraud_test.sum() + (float(COST_CONFIG["review_sensitivity"]) if REVIEW_ENABLED else 0.0) * reviewed_fraud_test.sum()) / fraud_mask_test.sum()),
                "expected_amount_weighted_recall": float(expected_detected_amount_test / fraud_amount_test),
                "review_count": int(np.sum(actions == "review")),
                "review_rate": float(np.mean(actions == "review")),
            })
    test_costs = pd.DataFrame(test_cost_rows)
    test_costs.to_csv(RESULTS_TABLES / "costos_test_sensibilidad.csv", index=False)
    display(test_costs)
    print("No vuelva a usar test para seleccionar o retocar modelos, costos o reglas.")

## 9. Lectura y límites

- La ventana `validation_policy` sirve para comparar/seleccionar configuraciones; no es una estimación final independiente.
- Test permanece sin predicciones mientras `RUN_FINAL_TEST=False`, aunque el manifiesto incluya su etiqueta.
- Los costos/ahorros (incluido ahorro absoluto y relativo frente al umbral fijo) solo se calculan con parámetros aprobados y unidades compatibles.
- El componente de `TransactionAmt` debe llamarse exposición/costo proxy, no pérdida bancaria real.
- El árbol ponderado aproxima costo binario en el ajuste y no representa la acción de revisión.
- Los resultados de este notebook no reemplazan pruebas automatizadas; estas no están presentes todavía en el repositorio.